# Baseline YOLO26s - Phát hiện vi phạm mũ bảo hiểm (DataSet2)

- Dataset: 3 lớp
  - `0: motorcycle`
  - `1: rider_with_helmet`
  - `2: rider_without_helmet`
- Giữ nguyên cách chia dữ liệu: Train: 1,563 ảnh | Valid: 140 ảnh | Test: 100 ảnh
- Notebook fine-tune pretrained `yolo26s.pt`, không dùng kiến trúc tùy biến.

In [ ]:
!pip install -U ultralytics
!pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
# Thiết lập dataset trên Google Colab

from google.colab import drive
import os
import gdown

drive.mount('/content/drive', force_remount=True)

# Giữ nguyên các đường dẫn dataset hiện tại.
!mkdir -p /content/DetectedHelmets/DataSet2
!mkdir -p /content/DetectedHelmets/Code
%cd /content/DetectedHelmets

if os.path.exists('dataset.rar'):
    os.remove('dataset.rar')

file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'dataset.rar'
gdown.download(url, output, quiet=False, fuzzy=True)

print('Đang giải nén dataset hiện tại...')
!unrar x -o+ {output} /content/DetectedHelmets/DataSet2/
print('Tải xuống và giải nén dataset hoàn tất.')

In [ ]:
import os

# Đường dẫn dataset trên Google Drive hiện tại; không thay đổi cách chia dữ liệu.
drive_dataset_path = '/content/drive/MyDrive/DataSet2'
local_dataset_path = '/content/DetectedHelmets/DataSet2'

# Dùng bản trên Drive nếu có, giống notebook ban đầu.
if os.path.exists(drive_dataset_path):
    if os.path.islink(local_dataset_path) or os.path.exists(local_dataset_path):
        !rm -rf {local_dataset_path}
    !ln -s "{drive_dataset_path}" "{local_dataset_path}"
    print(f'Đã liên kết dataset: {local_dataset_path} -> {drive_dataset_path}')
else:
    print(f'Không tìm thấy đường dẫn dataset trên Drive: {drive_dataset_path}')

## 1. Environment and GPU check

In [ ]:
import torch
import ultralytics

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print(f'Phiên bản Ultralytics: {ultralytics.__version__}')
print(f'Phiên bản PyTorch: {torch.__version__}')

if not torch.cuda.is_available():
    print('Hãy bật GPU runtime trong Colab trước khi train vì benchmark này yêu cầu device=0.')

## 2. Đường dẫn dataset và `data.yaml`

In [ ]:
import os
import yaml
from pathlib import Path

CURRENT_DIR = Path(os.getcwd()).resolve()
ROOT_DIR = CURRENT_DIR
while ROOT_DIR.name in ['Train', 'Code'] and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

# Giữ nguyên đường dẫn data.yaml ban đầu.
DATA_YAML_PATH = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'data.yaml'
MODEL_NAME = 'yolo26s.pt'
OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')

print(f'Thư mục project       : {ROOT_DIR}')
print(f'Đường dẫn data.yaml   : {DATA_YAML_PATH}')
print(f'Model pretrained      : {MODEL_NAME}')
print(f'Thư mục output        : {OUTPUT_DIR}')

with open(DATA_YAML_PATH, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])
if isinstance(class_names, dict):
    class_names = [class_names[key] for key in sorted(class_names, key=lambda item: int(item))]
CLASS_NAMES = list(class_names)

print('\n--- Cấu hình dataset ---')
print(f'Số lớp (nc) : {data_config.get("nc")}')
print(f'Tên lớp     : {CLASS_NAMES}')

## 3. Load và fine-tune baseline YOLO26s

Dùng nguyên kiến trúc pretrained YOLO26s: không thêm P2, attention, không sửa backbone/neck/head và không đổi loss.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26s.pt")
print('Đã load pretrained YOLO26s.')

In [ ]:
train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project=str(OUTPUT_DIR),
    name="helmet_yolo26s_dataset2",
    patience=15,
    save=True,
    save_period=10,
    plots=True,
    exist_ok=False
)

## 4. Validation trên tập validation

In [ ]:
import numpy as np

BEST_MODEL_PATH = OUTPUT_DIR / "helmet_yolo26s_dataset2" / "weights" / "best.pt"
print(f'Checkpoint tốt nhất: {BEST_MODEL_PATH}')
best_model = YOLO(
    OUTPUT_DIR / "helmet_yolo26s_dataset2" / "weights" / "best.pt"
)

metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=640,
    device=0,
    plots=True
)

precision = float(metrics.box.mp)
recall = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map5095 = float(metrics.box.map)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print('--- Kết quả validation (best.pt của YOLO26s) ---')
print(f'Precision (độ chính xác) : {precision:.4f}')
print(f'Recall (độ bao phủ)      : {recall:.4f}')
print(f'mAP50                    : {map50:.4f}')
print(f'mAP50-95                 : {map5095:.4f}')
print(f'F1                       : {f1:.4f}')

def print_per_class_metrics(split_metrics, split_name):
    print(f'\n--- Metric theo từng class ({split_name}) ---')
    for class_id, class_name in enumerate(CLASS_NAMES):
        p, r, ap50, ap = split_metrics.box.class_result(class_id)
        print(f'{class_id}: {class_name:24s} | Precision={p:.4f} | Recall={r:.4f} | AP50={ap50:.4f} | AP50-95={ap:.4f}')

        if class_name == 'rider_without_helmet':
            print(f'   Theo dõi rider_without_helmet -> Recall={r:.4f}, AP50={ap50:.4f}, AP50-95={ap:.4f}')

print_per_class_metrics(metrics, 'validation')

## 5. Đánh giá trên test (chỉ đánh giá, không dùng để chọn model)

In [ ]:
if data_config.get('test'):
    test_metrics = best_model.val(
        data=str(DATA_YAML_PATH),
        split="test",
        imgsz=640,
        device=0,
        plots=True
    )
    print('--- Kết quả test (best.pt của YOLO26s) ---')
    print(f'Precision (độ chính xác) : {test_metrics.box.mp:.4f}')
    print(f'Recall (độ bao phủ)      : {test_metrics.box.mr:.4f}')
    print(f'mAP50                    : {test_metrics.box.map50:.4f}')
    print(f'mAP50-95                 : {test_metrics.box.map:.4f}')
    print_per_class_metrics(test_metrics, 'test')
else:
    test_metrics = None
    print('data.yaml không khai báo test hợp lệ; đã bỏ qua đánh giá test.')

## 6. Hiển thị biểu đồ train và confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

exp_dir = OUTPUT_DIR / 'helmet_yolo26s_dataset2'
plot_files = [
    ('results.png', 'Kết quả train'),
    ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Confusion matrix chuẩn hóa'),
    ('PR_curve.png', 'Đường cong Precision-Recall'),
    ('F1_curve.png', 'Đường cong F1'),
    ('P_curve.png', 'Đường cong Precision'),
    ('R_curve.png', 'Đường cong Recall'),
]

for filename, title in plot_files:
    plot_path = exp_dir / filename
    if plot_path.exists():
        plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
        plt.imshow(Image.open(plot_path))
        plt.axis('off')
        plt.title(title)
        plt.show()
    else:
        print(f'Không tìm thấy (có thể phiên bản Ultralytics này không tạo): {plot_path}')

## 7. Prediction mẫu trên tập test

In [ ]:
import math
import random
from pathlib import Path

test_img_dir = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'test' / 'images'
test_images = sorted(
    [str(path) for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG') for path in test_img_dir.glob(pattern)]
)

if not test_images:
    raise FileNotFoundError(f'Không tìm thấy ảnh test trong: {test_img_dir}')

sample_images = random.sample(test_images, min(4, len(test_images)))
prediction_results = best_model.predict(
    source=sample_images,
    conf=0.4,
    imgsz=640,
    device=0,
    save=False,
    verbose=False
)

ncols = 2
nrows = max(1, math.ceil(len(prediction_results) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 6 * nrows))
axes = np.atleast_1d(axes).ravel()

for idx, result in enumerate(prediction_results):
    rendered_bgr = result.plot(conf=True)
    axes[idx].imshow(rendered_bgr[..., ::-1])
    axes[idx].axis('off')
    axes[idx].set_title(Path(result.path).name, fontsize=10)

for axis in axes[len(prediction_results):]:
    axis.axis('off')

plt.tight_layout()
plt.show()

## 8. Thông tin model tốt nhất: parameters và GFLOPs

In [ ]:
parameters = sum(parameter.numel() for parameter in best_model.model.parameters())
ultralytics_info = best_model.info(verbose=True, imgsz=640)

gflops = None
if isinstance(ultralytics_info, (tuple, list)) and ultralytics_info:
    try:
        gflops = float(ultralytics_info[-1])
    except (TypeError, ValueError):
        gflops = None
elif isinstance(ultralytics_info, dict):
    for key in ('GFLOPs', 'gflops', 'flops'):
        if key in ultralytics_info:
            try:
                gflops = float(ultralytics_info[key])
            except (TypeError, ValueError):
                pass
            break

print('Model summary đã được Ultralytics in ở phía trên.')
print(f'Số parameters: {parameters:,}')
print(f'GFLOPs       : {gflops:.4f}' if gflops is not None else 'GFLOPs       : phiên bản Ultralytics này không trả về giá trị')

## 9. Tốc độ inference trung bình trên ảnh test/validation

In [ ]:
import time

speed_images = test_images[:min(50, len(test_images))]
if not speed_images:
    valid_img_dir = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'valid' / 'images'
    speed_images = [str(path) for path in valid_img_dir.glob('*.*')][:50]

if not speed_images:
    raise FileNotFoundError('Không có ảnh test hoặc validation để đo tốc độ.')

# Warm-up cùng checkpoint tốt nhất trước khi đo.
for image_path in speed_images[:min(2, len(speed_images))]:
    best_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)

inference_times = []
for image_path in speed_images:
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    best_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_times.append(time.perf_counter() - start)

inference_time_ms = float(np.mean(inference_times) * 1000)
fps = 1000.0 / inference_time_ms if inference_time_ms else 0.0
print(f'Số ảnh đã đo            : {len(speed_images)}')
print(f'Thời gian inference TB : {inference_time_ms:.2f} ms/ảnh')
print(f'FPS                     : {fps:.2f}')

## 10. Bảng tổng hợp benchmark YOLO26s

In [ ]:
import pandas as pd

summary = pd.DataFrame([{
    'Model': 'YOLO26s',
    'Precision': precision,
    'Recall': recall,
    'F1': f1,
    'mAP50': map50,
    'mAP50-95': map5095,
    'Inference time (ms/image)': inference_time_ms,
    'FPS': fps,
    'Parameters': parameters,
    'GFLOPs': gflops,
}])
display(summary)

## 11. Export ONNX (tùy chọn)

In [ ]:
onnx_file = best_model.export(format='onnx', dynamic=True)
print(f'Đã export model ONNX tại: {onnx_file}')